# MatterGen analysis runner

This is the reusable postprocessing entry point for one `SYSTEM_ROOT`. The launcher copies this template to `<SYSTEM_ROOT>/analysis-run/mattergen_analysis.ipynb`, configures the VSBTools, MatterGen, and GRACE environments, and opens Jupyter. Edit the two YAML strings below, then run the cells in order.

The notebook discovers every homogeneous `non_guided/gen_*` and `repeated-guided/gen_*` tree below `SYSTEM_ROOT/raw-generations`. A cache key includes the YAML and all input files, so changing the scenario or generation inputs creates a fresh processed workspace automatically.

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import re
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import yaml
from IPython.display import display

import vsbtools
from vsbtools.materials_dataset.notebook_setup import (
    configure_notebook_external_environments,
)

external_environment = configure_notebook_external_environments(
    quiet_optional_imports=True,
)

VSBTOOLS_PACKAGE = Path(vsbtools.__file__).resolve().parent
CURRENT_DIR = Path.cwd().resolve()
DEFAULT_SYSTEM_ROOT = (
    CURRENT_DIR.parent if CURRENT_DIR.name == "analysis-run" else CURRENT_DIR
)
SYSTEM_ROOT = Path(
    os.environ.get("SYSTEM_ROOT", DEFAULT_SYSTEM_ROOT)
).expanduser().resolve()
RAW_ROOT = SYSTEM_ROOT / "raw-generations"
ANALYSIS_ROOT = SYSTEM_ROOT / "analysis-run"
FIGURE_ROOT = ANALYSIS_ROOT / "figures"
SCENARIO_PATH = ANALYSIS_ROOT / "scenario.yaml"
ANALYSIS_CONFIG_PATH = ANALYSIS_ROOT / "analysis.yaml"

ANALYSIS_ROOT.mkdir(parents=True, exist_ok=True)
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)

print("Python:", sys.executable)
print("VSBTools:", VSBTOOLS_PACKAGE)
for line in external_environment.summary_lines():
    print(line)
print("System root:", SYSTEM_ROOT)
print("Raw generations:", RAW_ROOT)
print("Analysis root:", ANALYSIS_ROOT)

## 1. Edit the scenario and analysis settings

The next two cells are ordinary YAML strings. Edit them in place and rerun from Section 1 onward. The scenario controls all postprocessing stages; analysis.yaml controls descriptors, losses, plots, and Pareto settings. A changed scenario or input tree receives a new processed cache key automatically.

In [ ]:
SCENARIO_YAML = """
version: 1

globals:
  toolkit_options:
    structure_parser:
      source_name: MatterGen
      batch_metadata_file: input_parameters.txt
    symmetry:
      a_sym_prec: 1.0e-2
      e_sym_prec: 1.0e-2
    similarity:
      tol_FP: 0.02
    estimator:
      default_model: grace
      force_gpu: 0
      grace_model: GRACE-3L-OMAT-large-ft-AM
    phase_diag: {}

stages:
  parse_raw:
    op: parse_raw
    needs: []
    params: {}

  check_min_dist:
    op: discard_close_atoms
    needs: [parse_raw]
    params: {}

  symmetrize_raw:
    op: symmetrize
    needs: [check_min_dist]
    params: {}

  estimate_all:
    op: estimate
    needs: [symmetrize_raw]
    params: {}

  poll_db:
    op: poll_db
    needs: [parse_raw]
    params:
      max_ehull: 0.5
      estimate_energies: true
      pref_db: op
      loader_kwargs:
        op:
          providers: [materials_project]
          timeout: 1500
      do_ehull_filtering: true
      do_deduplication: true
      tol_FP: 0.008

  deduplicate_all:
    op: deduplicate
    needs: [estimate_all]
    params: {}

  add_ref_all:
    op: merge_base_into_ref
    needs: [estimate_all, poll_db]
    params:
      base_parent: estimate_all
      ref_parent: poll_db
      merge: false

  add_ref_deduplicated:
    op: merge_base_into_ref
    needs: [deduplicate_all, poll_db]
    params:
      base_parent: deduplicate_all
      ref_parent: poll_db
      merge: false
""".strip() + "\n"

SCENARIO_PATH.write_text(SCENARIO_YAML, encoding="utf-8")
SCENARIO_CONFIG = yaml.safe_load(SCENARIO_YAML)
if not isinstance(SCENARIO_CONFIG, dict):
    raise TypeError("SCENARIO_YAML must contain a mapping at its root")
print("Scenario:", SCENARIO_PATH)

### 1.1 Analysis YAML

Use function names from the installed MatterGen bridge. With function: auto, a supported guided run supplies its descriptor and generated loss automatically. For an unguided run, choose a function and parameters explicitly. Species parameters may be written as element symbols or atomic numbers. An empty losses list keeps losses inferred from guided metadata; a populated list replaces those inferred losses.

In [ ]:
ANALYSIS_YAML = """
auto_infer_supported_guidance: true
summary_stage: add_ref_deduplicated

descriptor:
  function: auto
  column: null
  target: null
  axis_label: null
  params: {}

histogram:
  stage: symmetrize_raw
  reference_stage: poll_db
  kind: kde
  bins: 20
  max_value: null

pareto:
  fronts: 3
  trim_ehull: 0.3

losses: []

# Example manual descriptor:
# descriptor:
#   function: compute_mean_coordination
#   column: CN_[Pd,Ni]-H
#   target: 6
#   axis_label: "Mean CN([Pd,Ni]-H)"
#   params:
#     type_A: [Pd, Ni]
#     type_B: H
#     alpha: 3.0

# Example manual losses:
# losses:
#   - name: ranked_coordination
#     column: loss_ranked_coordination
#     params: {}
#     target:
#       margin: 0.05
#       temperature: 0.10
#       alpha: 2.0
#       cn_tolerance: 0.4
#       cn_temperature: 0.05
#       satisfaction_weight: 1.0
#       "[Pd,Ni]-H": 6
#   - name: group_coordination
#     column: loss_group_coordination
#     params: {}
#     target:
#       mode: huber
#       alpha: 3.0
#       "[Pd,Ni]-H": 6
""".strip() + "\n"

ANALYSIS_CONFIG_PATH.write_text(ANALYSIS_YAML, encoding="utf-8")
ANALYSIS_CONFIG = yaml.safe_load(ANALYSIS_YAML)
if not isinstance(ANALYSIS_CONFIG, dict):
    raise TypeError("ANALYSIS_YAML must contain a mapping at its root")
print("Analysis config:", ANALYSIS_CONFIG_PATH)

## 2. Discover generations and run the scenario

The generator keeps each scientific setting in one homogeneous gen_N directory, and every MatterGen invocation lives below it as run_N (a single run is run_1). The reader also accepts legacy batch_N trees. The hash below covers the editable scenario and the input tree, so a changed input set gets an independent processed cache.

In [ ]:
def _generation_roots(raw_root: Path) -> list[Path]:
    roots = []
    for mode in ("non_guided", "repeated-guided"):
        mode_root = raw_root / mode
        if not mode_root.is_dir():
            continue
        roots.extend(
            path
            for path in sorted(mode_root.glob("gen_*"))
            if path.is_dir()
        )
    return roots


def _input_records(roots: list[Path]) -> list[dict[str, object]]:
    records = []
    selected_names = {"input_parameters.txt", "generated_crystals.extxyz"}
    selected_suffixes = {".extxyz", ".zip"}
    for root in roots:
        for path in sorted(root.rglob("*")):
            if not path.is_file():
                continue
            if (
                path.name in selected_names
                or path.suffix.lower() in selected_suffixes
                or "POSCAR" in path.name
            ):
                stat = path.stat()
                records.append(
                    {
                        "path": path.relative_to(SYSTEM_ROOT).as_posix(),
                        "size": stat.st_size,
                        "mtime_ns": stat.st_mtime_ns,
                    }
                )
    return records


generation_roots = _generation_roots(RAW_ROOT)
if not generation_roots:
    raise RuntimeError(
        f"No generation roots found below {RAW_ROOT}. "
        "Run generate_mattergen.sh first."
    )

input_records = _input_records(generation_roots)
cache_payload = {
    "scenario": SCENARIO_YAML,
    "inputs": input_records,
}
RUN_ID = hashlib.sha256(
    json.dumps(cache_payload, sort_keys=True).encode("utf-8")
).hexdigest()[:12]
PROCESSED_ROOT = ANALYSIS_ROOT / "processed" / RUN_ID
PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)

print("Generation roots:")
print(*generation_roots, sep="\n")
print("Input files:", len(input_records))
print("Processed cache:", PROCESSED_ROOT)

In [ ]:
from vsbtools.materials_dataset.analysis.scenario_pipeline import (
    process_generation_dir,
)

repos = []
for generation_root in generation_roots:
    repo = process_generation_dir(
        generation_root,
        PROCESSED_ROOT,
        SCENARIO_PATH,
        batch_metadata_file="input_parameters.txt",
    )
    if repo is None:
        raise RuntimeError(
            f"Missing input_parameters.txt below {generation_root}"
        )
    repos.append(Path(repo.root))

system_roots = {repo.parent for repo in repos}
if len(system_roots) != 1:
    raise RuntimeError(
        "The discovered generation roots describe more than one chemical system: "
        + repr(sorted(str(path) for path in system_roots))
    )
PROCESSED_SYSTEM_ROOT = next(iter(system_roots))
print("Processed repositories:")
print(*repos, sep="\n")
print("Processed system:", PROCESSED_SYSTEM_ROOT)

## 3. Build summary tables and Pareto exports

The summary builder writes summary.csv, table.txt, Pareto-front CSVs, and POSCAR exports for every discovered generation repository. Guided metadata is reused for unguided repositories when it is supported. Manual descriptor and loss definitions in the analysis YAML remain available for every generation.

In [ ]:
from pymatgen.core import Element

from vsbtools.materials_dataset.analysis.guidance_statistics import (
    callables_from_ds,
    get_loss_fn,
    get_target_value_fn,
)
from vsbtools.materials_dataset.scripts.build_tables import (
    build_guidance_summary_for_processed_system,
    stage_datasets_from_repo,
)


def normalize_species_params(value):
    if isinstance(value, str):
        if re.fullmatch(r"[A-Z][a-z]?", value):
            return Element(value).Z
        return value
    if isinstance(value, (list, tuple)):
        return [normalize_species_params(item) for item in value]
    if isinstance(value, dict):
        return {key: normalize_species_params(item) for key, item in value.items()}
    return value


def _guided_parse_dataset():
    for repo in repos:
        parse_ds = stage_datasets_from_repo(repo).get("parse_raw")
        if parse_ds is None:
            continue
        guidance = parse_ds.metadata.get("batch_metadata", {}).get("guidance")
        if guidance not in (None, {}):
            return parse_ds
    return None


descriptor_cfg = dict(ANALYSIS_CONFIG.get("descriptor") or {})
force_gpu = int(
    (SCENARIO_CONFIG.get("globals", {})
     .get("toolkit_options", {})
     .get("estimator", {})
     .get("force_gpu", 0))
)
inferred_callables = {}
inferred_targets = {}
inferred_guidance_name = None
guided_parse_ds = _guided_parse_dataset()

if ANALYSIS_CONFIG.get("auto_infer_supported_guidance", True) and guided_parse_ds is not None:
    try:
        (
            inferred_callables,
            inferred_targets,
            inferred_guidance_name,
        ) = callables_from_ds(guided_parse_ds, force_gpu=force_gpu)
    except (AssertionError, KeyError, ValueError) as exc:
        print("Automatic guidance inference was skipped:", exc)

report_callables = dict(inferred_callables)
raw_losses = ANALYSIS_CONFIG.get("losses") or []
if raw_losses:
    report_callables = {
        name: fn for name, fn in report_callables.items()
        if not str(name).startswith("loss_")
    }
requested_column = descriptor_cfg.get("column")
requested_function = descriptor_cfg.get("function")
descriptor_name = None
descriptor_fn = None
automatic_descriptor_names = [
    name for name in inferred_callables if not name.startswith("loss_")
]

if requested_column in inferred_callables:
    descriptor_name = requested_column
elif (
    requested_function not in (None, "auto")
    and requested_function in inferred_callables
):
    descriptor_name = requested_function
elif requested_function in (None, "auto") and len(automatic_descriptor_names) == 1:
    descriptor_name = automatic_descriptor_names[0]
elif requested_function in (None, "auto") and automatic_descriptor_names:
    raise ValueError(
        "Multiple descriptors were inferred from guided metadata; set "
        "descriptor.column in ANALYSIS_YAML to one of "
        f"{automatic_descriptor_names!r}"
    )

if descriptor_name is not None:
    descriptor_fn = inferred_callables[descriptor_name]
    inferred_target = inferred_targets.get(descriptor_name)
else:
    requested_function = (
        requested_function if requested_function not in (None, "auto") else "volume_pa"
    )
    descriptor_name = requested_column or requested_function
    descriptor_params = normalize_species_params(
        descriptor_cfg.get("params") or {}
    )
    descriptor_fn = get_target_value_fn(
        requested_function,
        force_gpu=force_gpu,
        **descriptor_params,
    )
    inferred_target = None

configured_target = descriptor_cfg.get("target")
descriptor_target = (
    inferred_target if configured_target is None else configured_target
)
descriptor_label = descriptor_cfg.get("axis_label") or descriptor_name
report_callables.setdefault(descriptor_name, descriptor_fn)

if isinstance(raw_losses, dict):
    raw_losses = [
        dict(value or {}, name=name)
        for name, value in raw_losses.items()
    ]
for loss_cfg in raw_losses:
    loss_cfg = dict(loss_cfg)
    loss_name = loss_cfg.get("name") or loss_cfg.get("function")
    if not loss_name:
        raise ValueError(f"Loss entry has no name: {loss_cfg!r}")
    loss_column = loss_cfg.get("column") or f"loss_{loss_name}"
    loss_params = normalize_species_params(loss_cfg.get("params") or {})
    loss_target = loss_cfg.get("target")
    if loss_target is None:
        loss_target = loss_params
    report_callables[loss_column] = get_loss_fn(
        loss_name,
        force_gpu=force_gpu,
        target=loss_target,
        **loss_params,
    )

summary_stage = ANALYSIS_CONFIG.get("summary_stage", "add_ref_deduplicated")
pareto_cfg = dict(ANALYSIS_CONFIG.get("pareto") or {})
summary_report = build_guidance_summary_for_processed_system(
    PROCESSED_SYSTEM_ROOT,
    target_stages=[summary_stage],
    auto_ref_stages=True,
    callables=report_callables,
    max_pareto_front=int(pareto_cfg.get("fronts", 3)),
    return_report=True,
)
report_callables = dict(summary_report)

print("Guidance:", inferred_guidance_name or "manual/unguided")
print("Descriptor:", descriptor_name, "target:", descriptor_target)
print("Summary stage:", summary_stage)
print(summary_report)

## 4. Plot the configured descriptor

The histogram cell supports KDE and multihistogram output. Set histogram.kind to kde or histogram in the YAML. It compares all processed generation repositories and the reference dataset when the selected scenario stage contains one.

In [ ]:
from vsbtools.materials_dataset.analysis.guidance_statistics import (
    calculate_values,
    collect_stage_dataset_dict,
    histo_data_collection,
    plot_multi_kde,
    plot_multihistogram,
)

hist_cfg = dict(ANALYSIS_CONFIG.get("histogram") or {})
hist_stage = hist_cfg.get("stage", "symmetrize_raw")
reference_stage = hist_cfg.get("reference_stage", "poll_db")
plot_kind = str(hist_cfg.get("kind", "kde")).lower()
hist_datasets = collect_stage_dataset_dict(
    repos,
    stage=hist_stage,
    ref_stage=reference_stage,
)
if not hist_datasets:
    raise RuntimeError(f"No datasets found for histogram stage {hist_stage!r}")

max_value = hist_cfg.get("max_value")
if plot_kind == "kde":
    hist_values = calculate_values(
        hist_datasets,
        fn=descriptor_fn,
        filter_max_el=False,
    )
    fig, ax = plot_multi_kde(
        hist_values,
        target=descriptor_target,
        max_value=max_value,
        simplified_legend=True,
    )
elif plot_kind in {"histogram", "hist"}:
    histogram_data = histo_data_collection(
        hist_datasets,
        fn=descriptor_fn,
        filter_max_el=False,
        auto_adjust_bins=True,
        n_bins=int(hist_cfg.get("bins", 20)),
    )
    fig, ax = plot_multihistogram(
        histogram_data,
        target=descriptor_target,
        max_bincenter=max_value,
        show_gaussian=True,
        simplified_legend=True,
    )
else:
    raise ValueError("histogram.kind must be 'kde' or 'histogram'")

ax.set_xlabel(descriptor_label)
safe_descriptor = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(descriptor_name)).strip("_")
histogram_path = FIGURE_ROOT / (
    f"{SYSTEM_ROOT.name}_{safe_descriptor}_{plot_kind}.pdf"
)
fig.savefig(histogram_path, bbox_inches="tight", pad_inches=0.1)
display(fig)
plt.close(fig)
print("Saved:", histogram_path)

## 5. Plot Pareto fronts

For every configured loss, the summary stage creates Pareto-front CSVs. This cell renders each available loss against e_hull/at for every generation repository and saves the figures below analysis-run/figures.

In [ ]:
from vsbtools.materials_dataset.analysis.pareto_fronts import plot_pareto

pareto_fronts = int(pareto_cfg.get("fronts", 3))
trim_ehull = pareto_cfg.get("trim_ehull")
loss_columns = [
    name for name in report_callables
    if str(name).startswith("loss_")
]
loss_prefixes = [""] if len(loss_columns) <= 1 else [
    name.removeprefix("loss_") + "_"
    for name in loss_columns
]

pareto_paths = []
for repo_index, repo in enumerate(repos, start=1):
    datasets = stage_datasets_from_repo(repo)
    stage_dataset = datasets.get(summary_stage)
    if stage_dataset is None or stage_dataset.base_path is None:
        print("Skipping Pareto plots; stage is missing:", repo)
        continue
    stage_dir = Path(stage_dataset.base_path)
    for loss_column, prefix in zip(loss_columns, loss_prefixes):
        if not (stage_dir / f"{prefix}pf_1.csv").exists():
            print(f"No Pareto CSV for {loss_column} in {stage_dir}")
            continue
        ax = plot_pareto(
            stage_dir,
            col1=loss_column,
            col2="e_hull/at",
            trim_col2=trim_ehull,
            n_fronts=pareto_fronts,
            prefix=prefix,
            article_axes=True,
            show_title=False,
        )
        pareto_path = FIGURE_ROOT / (
            f"{SYSTEM_ROOT.name}_repo_{repo_index}_"
            f"{safe_descriptor}_{prefix}pareto.pdf"
        )
        ax.figure.savefig(pareto_path, bbox_inches="tight", pad_inches=0.1)
        display(ax.figure)
        plt.close(ax.figure)
        pareto_paths.append(pareto_path)
        print("Saved:", pareto_path)

if not loss_columns:
    print("No loss callables configured; Pareto export was skipped.")

## 6. Record the run

The manifest makes the analysis reproducible: it records the system path, cache key, YAML settings, discovered inputs, processed repositories, callable names, and generated figures.

In [ ]:
analysis_manifest = {
    "python": sys.executable,
    "vsbtools_package": str(VSBTOOLS_PACKAGE),
    "mattergen_python_path": os.environ.get("MATTERGEN_PYTHON_PATH"),
    "mattergen_site_packages": os.environ.get("SCOUT_MATTER_SITE_PACKAGES"),
    "grace_python": os.environ.get("GRACE_PYTHON"),
    "system_root": str(SYSTEM_ROOT),
    "raw_root": str(RAW_ROOT),
    "scenario_path": str(SCENARIO_PATH),
    "analysis_config_path": str(ANALYSIS_CONFIG_PATH),
    "scenario_sha256": hashlib.sha256(SCENARIO_YAML.encode("utf-8")).hexdigest(),
    "analysis_config_sha256": hashlib.sha256(ANALYSIS_YAML.encode("utf-8")).hexdigest(),
    "processed_cache": str(PROCESSED_ROOT),
    "input_records": input_records,
    "generation_roots": [str(path) for path in generation_roots],
    "processed_repositories": [str(path) for path in repos],
    "descriptor": {
        "name": descriptor_name,
        "target": descriptor_target,
        "label": descriptor_label,
    },
    "callables": sorted(str(name) for name in report_callables),
    "figures": [str(path) for path in [histogram_path, *pareto_paths]],
}
ANALYSIS_MANIFEST_PATH = ANALYSIS_ROOT / "analysis_manifest.json"
ANALYSIS_MANIFEST_PATH.write_text(
    json.dumps(analysis_manifest, indent=2, default=str),
    encoding="utf-8",
)
print("Analysis complete.")
print("Manifest:", ANALYSIS_MANIFEST_PATH)
print("Figures:", FIGURE_ROOT)